# EP04 — Value at Risk & Expected Shortfall
**Quantifaya · Classical Quantitative Finance Series · Episode 4**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Godwin-88/quantifire-web/blob/main/public/notebooks/ep04-value-at-risk.ipynb)

> **Learning objective:** Derive three VaR calculation methods, understand why VaR ignores tail severity, implement Expected Shortfall (CVaR), and build a complete risk reporting system.

**Companion post:** [quantifaya.com/blog/ep04-value-at-risk-how-much-lose-bad-day](https://quantifaya.com/blog/ep04-value-at-risk-how-much-lose-bad-day)

---
*Quantifaya research notebooks are provided for educational purposes only. Nothing here constitutes financial advice.*

## Learning Objectives

By the end of this notebook you will be able to:

- **Calculate** VaR using three methods: Historical, Normal (parametric), and Student-t (fat-tail aware)
- **Compare** VaR estimates across methods and understand why Normal VaR underestimates tail risk
- **Compute** Expected Shortfall (CVaR) — the tail-aware metric that VaR misses
- **Explain** why CVaR is a coherent risk measure while VaR is not
- **Scale** VaR across time horizons using √T and understand when this fails
- **Decompose** portfolio VaR into component contributions by asset
- **Backtest** VaR models using the Kupiec test
- **Adjust** VaR for DeFi-specific risks (liquidation cascades, smart contract exploits)

## Mathematical Prerequisites

### Value at Risk Definition

$$\text{VaR}_\alpha = -\inf\{r : F(r) \leq \alpha\} = -q_\alpha$$

Where $q_\alpha$ is the $\alpha$-quantile of the return distribution.

### Historical VaR

$$\text{VaR}_\alpha^{\text{hist}} = -\hat{q}_\alpha(r)$$

### Parametric VaR (Normal)

$$\text{VaR}_\alpha^{\text{norm}} = -(\mu - z_\alpha \cdot \sigma)$$

Where $z_{0.05} = 1.645$ (95% VaR), $z_{0.01} = 2.326$ (99% VaR).

### Parametric VaR (Student-t)

$$\text{VaR}_\alpha^{t} = -(\mu - t_{\nu,\alpha} \cdot \sigma)$$

### Expected Shortfall (CVaR)

$$\text{CVaR}_\alpha = -\mathbb{E}[r \mid r < -\text{VaR}_\alpha]$$

### √T Scaling (under iid assumption)

$$\text{VaR}_T = \text{VaR}_1 \cdot \sqrt{T}$$

## Setup: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import stats
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries loaded successfully.")

## 1. VaR Calculation Functions

Three methods to compute Value at Risk:

In [ ]:
def var_historical(returns, alpha=0.05):
    """
    Historical VaR: empirical quantile.
    
    Args:
        returns: Array of historical returns
        alpha: Confidence level (0.05 for 95% VaR)
    
    Returns:
        VaR as a positive number (loss amount)
    """
    return -np.percentile(returns, alpha * 100)


def var_normal(returns, alpha=0.05):
    """
    Parametric VaR assuming normal distribution.
    """
    mu = np.mean(returns)
    sigma = np.std(returns)
    z_alpha = stats.norm.ppf(alpha)
    return -(mu + z_alpha * sigma)


def var_student_t(returns, alpha=0.05):
    """
    Parametric VaR assuming Student-t distribution.
    """
    mu = np.mean(returns)
    sigma = np.std(returns)
    
    # Fit t-distribution to get degrees of freedom
    df, loc, scale = stats.t.fit(returns)
    
    # VaR from fitted t-distribution
    t_quantile = stats.t.ppf(alpha, df, loc, scale)
    return -t_quantile


def expected_shortfall(returns, alpha=0.05):
    """
    Expected Shortfall (CVaR): expected loss given VaR is breached.
    
    Also called: Conditional VaR, Average VaR, Tail VaR
    """
    var = var_historical(returns, alpha)
    tail_losses = returns[returns <= -var]
    return -tail_losses.mean() if len(tail_losses) > 0 else var


def estimate_t_dof(returns):
    """Estimate degrees of freedom for t-distribution fit."""
    params = stats.t.fit(returns)
    return params[0]


print("VaR calculation functions defined.")

## 2. Complete VaR Reporting Suite

In [ ]:
def complete_var_report(returns, portfolio_value=1_000_000, alpha=0.05, name="Portfolio"):
    """
    Complete VaR reporting suite.
    """
    report = {
        'Portfolio': name,
        'Portfolio Value': portfolio_value,
        'Confidence Level': (1 - alpha) * 100,
        'Observations': len(returns),
        
        # VaR estimates (daily)
        'Historical VaR (daily)': var_historical(returns, alpha),
        'Normal VaR (daily)': var_normal(returns, alpha),
        'Student-t VaR (daily)': var_student_t(returns, alpha),
        
        # Expected Shortfall
        'Expected Shortfall (CVaR)': expected_shortfall(returns, alpha),
        
        # Distribution diagnostics
        'Mean Return (daily)': np.mean(returns),
        'Volatility (daily)': np.std(returns),
        'Skewness': stats.skew(returns),
        'Excess Kurtosis': stats.kurtosis(returns),
        
        # t-distribution fit
        't-distribution df': estimate_t_dof(returns),
    }
    
    # 10-day VaR (square-root-of-time scaling)
    report['Historical VaR (10-day)'] = var_historical(returns, alpha) * np.sqrt(10)
    report['Normal VaR (10-day)'] = var_normal(returns, alpha) * np.sqrt(10)
    
    return pd.Series(report)


def print_var_report(report):
    """Pretty-print a VaR report."""
    print(f"\n{'='*55}")
    print(f"  VaR REPORT: {report['Portfolio']}")
    print(f"{'='*55}")
    print(f"  Portfolio Value:      ${report['Portfolio Value']:>12,.0f}")
    print(f"  Confidence Level:     {report['Confidence Level']:>12.0f}%")
    print(f"  Observations:         {report['Observations']:>12}")
    print(f"  ─" * 20)
    print(f"  Historical VaR (1d):  ${report['Historical VaR (daily)'] * report['Portfolio Value']:>12,.0f}")
    print(f"  Normal VaR (1d):      ${report['Normal VaR (daily)'] * report['Portfolio Value']:>12,.0f}")
    print(f"  Student-t VaR (1d):   ${report['Student-t VaR (daily)'] * report['Portfolio Value']:>12,.0f}")
    print(f"  Expected Shortfall:   ${report['Expected Shortfall (CVaR)'] * report['Portfolio Value']:>12,.0f}")
    print(f"  ─" * 20)
    print(f"  Historical VaR (10d): ${report['Historical VaR (10-day)'] * report['Portfolio Value']:>12,.0f}")
    print(f"  Normal VaR (10d):     ${report['Normal VaR (10-day)'] * report['Portfolio Value']:>12,.0f}")
    print(f"  ─" * 20)
    print(f"  Mean Return (daily):  {report['Mean Return (daily)']:>12.4%}")
    print(f"  Volatility (daily):   {report['Volatility (daily)']:>12.4%}")
    print(f"  Skewness:             {report['Skewness']:>12.3f}")
    print(f"  Excess Kurtosis:      {report['Excess Kurtosis']:>12.3f}")
    print(f"  t-distribution df:    {report['t-distribution df']:>12.2f}")
    print(f"{'='*55}")
    
    # Warning flags
    hist_var = report['Historical VaR (daily)']
    norm_var = report['Normal VaR (daily)']
    if norm_var < hist_var * 0.9:
        print(f"\n  ⚠️  WARNING: Normal VaR underestimates risk by {((hist_var - norm_var) / hist_var) * 100:.1f}%")
        print("     → Fat tails detected. Use Student-t or Historical VaR.")
    
    es = report['Expected Shortfall (CVaR)']
    if es > hist_var * 1.2:
        print(f"\n  ⚠️  WARNING: CVaR is {es / hist_var:.1f}x VaR → severe tail risk")
        print("     → Losses beyond VaR threshold are significantly worse.")

## 3. Compare VaR Methods on Fat-Tailed vs Normal Returns

In [ ]:
np.random.seed(42)
n_days = 500

# Simulate fat-tailed returns (t-distribution with df=5)
returns_t = stats.t.rvs(df=5, loc=0.0003, scale=0.015, size=n_days)

# Generate normal returns for comparison
returns_norm = np.random.normal(0.0003, 0.015, n_days)

s_t = pd.Series(returns_t)
s_norm = pd.Series(returns_norm)

print("\n" + "="*60)
print("VaR METHOD COMPARISON")
print("="*60)

report_t = complete_var_report(s_t, portfolio_value=1_000_000, alpha=0.05, name="Fat-Tailed (t-dist, df=5)")
report_norm = complete_var_report(s_norm, portfolio_value=1_000_000, alpha=0.05, name="Normal Returns")

print_var_report(report_t)
print_var_report(report_norm)

# Visualize return distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, s, title in zip(axes, [s_t, s_norm], 
                         ['Fat-Tailed Returns (t-dist, df=5)', 'Normal Returns']):
    ax.hist(s * 100, bins=50, alpha=0.7, edgecolor='black', density=True, color='steelblue')
    
    # Overlay normal PDF
    x = np.linspace(s.min(), s.max(), 100)
    ax.plot(x * 100, stats.norm.pdf(x, s.mean(), s.std()), 
            'r--', linewidth=2, label='Normal PDF')
    
    # Mark VaR thresholds
    var_hist = var_historical(s)
    var_norm = var_normal(s)
    ax.axvline(-var_hist * 100, color='red', linestyle='-', linewidth=2, 
               label=f'Historical VaR: {var_hist:.2%}')
    ax.axvline(-var_norm * 100, color='orange', linestyle='--', linewidth=2, 
               label=f'Normal VaR: {var_norm:.2%}')
    
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Key Insight ===")
print("Normal VaR is dangerously optimistic for fat-tailed returns.")
print("Student-t VaR correctly captures tail risk when df < 6.")

## 4. VaR vs CVaR: The Tail Severity Problem

Demonstrate why VaR alone is insufficient:

In [ ]:
# Create two strategies with identical VaR but different tail severity
np.random.seed(300)
n_days = 10000

# Strategy A: Bounded losses (95% VaR = $10k, max loss = $10k)
returns_A = np.random.normal(0.0005, 0.008, n_days)
# Cap losses at VaR level
var_95_A = var_historical(returns_A, 0.05)
returns_A = np.maximum(returns_A, -var_95_A)

# Strategy B: Same VaR but catastrophic tail
returns_B = np.random.normal(0.0005, 0.008, n_days)
returns_B = np.maximum(returns_B, -var_95_A)  # Same VaR threshold
# Add catastrophic tail: 0.5% chance of 50x VaR loss
catastrophe_days = np.random.choice(n_days, size=int(n_days * 0.005), replace=False)
returns_B[catastrophe_days] = -var_95_A * 50

s_A = pd.Series(returns_A)
s_B = pd.Series(returns_B)

print("\n" + "="*60)
print("VaR vs CVaR: The Tail Severity Problem")
print("="*60)

report_A = complete_var_report(s_A, portfolio_value=1_000_000, alpha=0.05, name="Strategy A (Bounded)")
report_B = complete_var_report(s_B, portfolio_value=1_000_000, alpha=0.05, name="Strategy B (Catastrophic Tail)")

print_var_report(report_A)
print_var_report(report_B)

# Summary comparison
print("\n=== Comparison Summary ===")
print(f"{'Metric':<25} {'Strategy A':<20} {'Strategy B':<20}")
print("-" * 65)
print(f"{'95% VaR':<25} ${report_A['Historical VaR (daily)'] * 1e6:>12,.0f}    ${report_B['Historical VaR (daily)'] * 1e6:>12,.0f}")
print(f"{'Expected Shortfall':<25} ${report_A['Expected Shortfall (CVaR)'] * 1e6:>12,.0f}    ${report_B['Expected Shortfall (CVaR)'] * 1e6:>12,.0f}")
print(f"{'CVaR/VaR Ratio':<25} {report_A['Expected Shortfall (CVaR)'] / report_A['Historical VaR (daily)']:>12.2f}x    {report_B['Expected Shortfall (CVaR)'] / report_B['Historical VaR (daily)']:>12.2f}x")
print(f"{'Max Loss':<25} ${abs(s_A.min()) * 1e6:>12,.0f}    ${abs(s_B.min()) * 1e6:>12,.0f}")

print("\n>>> Both strategies have IDENTICAL VaR.")
print(">>> But Strategy B's CVaR reveals catastrophic tail risk.")
print(">>> VaR alone would not distinguish them!")

## 5. VaR Scaling: √T Rule and Its Failures

In [ ]:
# Demonstrate √T scaling and when it fails
np.random.seed(500)
n_days = 1000

# Case 1: IID returns (√T works)
returns_iid = np.random.normal(0.0005, 0.015, n_days)

# Case 2: Volatility clustering (GARCH-like, √T underestimates)
returns_cluster = np.zeros(n_days)
returns_cluster[0] = np.random.normal(0.0005, 0.015)
for t in range(1, n_days):
    # Volatility depends on previous day's magnitude
    vol_t = 0.01 + 0.5 * abs(returns_cluster[t-1]) + 0.3 * 0.015
    returns_cluster[t] = np.random.normal(0.0005, vol_t)

s_iid = pd.Series(returns_iid)
s_cluster = pd.Series(returns_cluster)

print("\n" + "="*60)
print("VaR SCALING: √T Rule vs Reality")
print("="*60)

horizons = [1, 5, 10, 21]  # 1-day, 1-week, 2-week, 1-month

print(f"\n{'Horizon':<12} {'IID 1-day':<15} {'IID √T-scaled':<18} {'Cluster 1-day':<18} {'Cluster √T-scaled':<20}")
print("-" * 83)

for h in horizons:
    # √T scaled VaR
    iid_scaled = var_historical(s_iid) * np.sqrt(h)
    cluster_scaled = var_historical(s_cluster) * np.sqrt(h)
    
    print(f"{h:>4} day{'s' if h > 1 else ' ':<5} ${var_historical(s_iid) * 1e6:>10,.0f}     ${iid_scaled * 1e6:>12,.0f}       ${var_historical(s_cluster) * 1e6:>10,.0f}       ${cluster_scaled * 1e6:>14,.0f}")

# Plot scaling comparison
fig, ax = plt.subplots(figsize=(12, 7))

h_vals = np.array(horizons)
iid_base = var_historical(s_iid)
cluster_base = var_historical(s_cluster)

# Theoretical √T scaling lines
h_fine = np.linspace(1, 30, 100)
ax.plot(h_fine, iid_base * np.sqrt(h_fine), 'b--', linewidth=2, label='√T Scaling (IID)')
ax.plot(h_fine, cluster_base * np.sqrt(h_fine), 'r--', linewidth=2, label='√T Scaling (Clustered)')

# Actual multi-day VaR (computed directly)
def multi_day_var(returns, horizon, alpha=0.05):
    """Compute actual multi-day VaR by summing consecutive returns."""
    n = len(returns)
    multi_day_returns = []
    for i in range(n - horizon):
        multi_day_returns.append(np.sum(returns[i:i+horizon]))
    return -np.percentile(multi_day_returns, alpha * 100)

actual_iid = [multi_day_var(s_iid.values, h) for h in horizons]
actual_cluster = [multi_day_var(s_cluster.values, h) for h in horizons]

ax.scatter(horizons, actual_iid, c='blue', s=100, zorder=5, label='Actual Multi-day VaR (IID)')
ax.scatter(horizons, actual_cluster, c='red', s=100, zorder=5, label='Actual Multi-day VaR (Clustered)')

ax.set_xlabel('Time Horizon (days)', fontsize=14)
ax.set_ylabel('VaR', fontsize=14)
ax.set_title('√T Scaling: Works for IID, Fails with Volatility Clustering', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Key Takeaway ===")
print("√T scaling assumes independent returns.")
print("During volatility clustering (crises), actual multi-day VaR > √T estimate.")

## 6. Component VaR: Decomposing Risk by Asset

For a multi-asset portfolio, identify which positions contribute most to total VaR:

In [ ]:
def component_var_normal(weights, cov_matrix, portfolio_value, alpha=0.05):
    """
    Component VaR decomposition (normal assumption).
    
    CVaR_i = w_i * (Σw)_i / σ_p * z_α
    """
    sigma_p = np.sqrt(weights @ cov_matrix @ weights)
    z_alpha = stats.norm.ppf(alpha)
    
    # Marginal contribution to risk
    marginal_risk = (cov_matrix @ weights) / sigma_p
    
    # Component VaR
    component_var = weights * marginal_risk * z_alpha * portfolio_value
    
    return {
        'Total VaR': abs(np.sum(component_var)),
        'Components': component_var,
        'Percent Contribution': abs(component_var) / abs(np.sum(component_var)) * 100
    }


# Define a 5-asset portfolio
assets = ['SPY', 'TLT', 'GLD', 'VNQ', 'EEM']
weights = np.array([0.35, 0.25, 0.15, 0.15, 0.10])

# Annual parameters
vols = np.array([0.18, 0.08, 0.15, 0.20, 0.22])
corr = np.array([
    [1.00, -0.35,  0.05,  0.72,  0.65],
    [-0.35,  1.00,  0.28, -0.20, -0.28],
    [ 0.05,  0.28,  1.00,  0.08,  0.10],
    [ 0.72, -0.20,  0.08,  1.00,  0.55],
    [ 0.65, -0.28,  0.10,  0.55,  1.00],
])
cov_annual = np.outer(vols, vols) * corr
cov_daily = cov_annual / 252

portfolio_value = 1_000_000
result = component_var_normal(weights, cov_daily, portfolio_value, alpha=0.05)

print("\n" + "="*60)
print("COMPONENT VaR DECOMPOSITION")
print("="*60)

print(f"\nTotal 95% Daily VaR: ${result['Total VaR']:,.0f}")
print("\nAsset Breakdown:")
print(f"{'Asset':<8} {'Weight':<10} {'Component VaR':<18} {'% Contribution':<15}")
print("-" * 51)

for asset, w, cv, pct in zip(assets, weights, result['Components'], result['Percent Contribution']):
    print(f"{asset:<8} {w:>7.1%}    ${abs(cv):>10,.0f}      {pct:>6.1f}%")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart of contribution
axes[0].pie(result['Percent Contribution'], labels=assets, autopct='%1.1f%%',
            colors=plt.cm.Set2(np.linspace(0, 1, len(assets))))
axes[0].set_title('VaR Contribution by Asset', fontsize=14)

# Bar chart
axes[1].bar(assets, result['Percent Contribution'], 
            color=plt.cm.Set2(np.linspace(0, 1, len(assets))))
axes[1].set_xlabel('Asset', fontsize=12)
axes[1].set_ylabel('% of Total VaR', fontsize=12)
axes[1].set_title('Component VaR Contribution', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')

# Highlight concentration risk
max_contrib = result['Percent Contribution'].max()
max_asset = assets[np.argmax(result['Percent Contribution'])]
if max_contrib > 40:
    axes[1].text(0.5, 0.9, f'⚠️ {max_asset} contributes {max_contrib:.0f}% → Concentration risk!',
                 transform=axes[1].transAxes, fontsize=11, color='red', ha='center',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("CONCENTRATION RISK ANALYSIS")
print(f"{'='*60}")
print(f"Highest contributor: {max_asset} ({max_contrib:.1f}%)")
if max_contrib > 40:
    print(f"⚠️  WARNING: {max_asset} contributes >40% of total VaR!")
    print("   → Portfolio has hidden concentration despite diversified weights.")
else:
    print("✅ Risk is reasonably diversified across assets.")

## 7. VaR Backtesting: The Kupiec Test

In [ ]:
def var_backtest_kupiec(actual_returns, var_estimates, alpha=0.05):
    """
    Kupiec test for VaR model validation.
    
    Tests whether the observed exception rate matches the expected rate.
    """
    T = len(actual_returns)
    exceptions = actual_returns < -var_estimates
    N = np.sum(exceptions)
    
    p = alpha  # Expected exception rate
    p_hat = N / T  # Observed exception rate
    
    if N == 0 or N == T:
        return {'status': 'INVALID', 'reason': 'Zero or all exceptions'}
    
    # Likelihood ratio
    lr_uc = -2 * (
        np.log((1-p)**(T-N) * p**N) -
        np.log((1-p_hat)**(T-N) * p_hat**N)
    )
    
    # p-value from chi-squared distribution
    p_value = 1 - stats.chi2.cdf(lr_uc, df=1)
    
    # Decision
    if p_value < 0.05:
        status = 'REJECT'  # Model is invalid
    elif p_hat > p * 1.5:
        status = 'AMBER'   # Too many exceptions
    else:
        status = 'PASS'
    
    return {
        'Status': status,
        'Expected Exceptions': p * T,
        'Actual Exceptions': N,
        'Exception Rate': p_hat,
        'LR Statistic': lr_uc,
        'p-value': p_value,
    }


# Backtest our VaR models
np.random.seed(600)
n_days = 500
returns_test = stats.t.rvs(df=5, loc=0.0003, scale=0.015, size=n_days)

# Compute VaR estimates for each day (rolling window)
window = 100
var_estimates_hist = np.full(n_days, np.nan)
var_estimates_norm = np.full(n_days, np.nan)
var_estimates_t = np.full(n_days, np.nan)

for t in range(window, n_days):
    window_returns = returns_test[t-window:t]
    var_estimates_hist[t] = var_historical(window_returns, 0.05)
    var_estimates_norm[t] = var_normal(window_returns, 0.05)
    var_estimates_t[t] = var_student_t(window_returns, 0.05)

# Remove NaN values
valid_mask = ~np.isnan(var_estimates_hist)
actual_valid = returns_test[valid_mask]
var_hist_valid = var_estimates_hist[valid_mask]
var_norm_valid = var_estimates_norm[valid_mask]
var_t_valid = var_estimates_t[valid_mask]

print("\n" + "="*60)
print("VaR BACKTESTING: KUPIEC TEST")
print("="*60)

print(f"\nTest period: {len(actual_valid)} days")
print(f"Expected exceptions (5%): {0.05 * len(actual_valid):.0f}")
print()

for method, var_est, label in [
    (var_hist_valid, actual_valid, "Historical VaR"),
    (var_norm_valid, actual_valid, "Normal VaR"),
    (var_t_valid, actual_valid, "Student-t VaR")
]:
    result = var_backtest_kupiec(actual_valid, var_est, alpha=0.05)
    
    print(f"--- {label} ---")
    print(f"  Status:              {result['Status']}")
    print(f"  Expected Exceptions: {result['Expected Exceptions']:.0f}")
    print(f"  Actual Exceptions:   {result['Actual Exceptions']}")
    print(f"  Exception Rate:      {result['Exception Rate']:.2%}")
    print(f"  LR Statistic:        {result['LR Statistic']:.3f}")
    print(f"  p-value:             {result['p-value']:.4f}")
    print()

# Plot exceptions over time
fig, ax = plt.subplots(figsize=(14, 6))

t_vals = np.arange(len(actual_valid))
exceptions_hist = actual_valid < -var_hist_valid
exceptions_norm = actual_valid < -var_norm_valid

ax.plot(t_vals, actual_valid * 100, 'k-', alpha=0.5, linewidth=1, label='Daily Return (%)')
ax.plot(t_vals, -var_hist_valid * 100, 'b-', linewidth=2, label='Historical VaR (5%)')
ax.plot(t_vals, -var_norm_valid * 100, 'r--', linewidth=2, label='Normal VaR (5%)')

# Mark exceptions
ax.scatter(t_vals[exceptions_hist], actual_valid[exceptions_hist] * 100, 
           c='blue', s=50, zorder=5, label='Historical VaR Breaches')

ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Return (%)', fontsize=12)
ax.set_title('VaR Backtesting: Exception Tracking', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. DeFi-Specific VaR Adjustments

In [ ]:
def defi_var_adjustment(normal_var, liquidation_multiplier=1.5, 
                        exploit_prob_annual=0.005, exploit_loss=1.0):
    """
    Adjust VaR for DeFi-specific risks.
    
    Args:
        normal_var: Standard market VaR
        liquidation_multiplier: Factor for liquidation cascade risk (1.5-2.0x)
        exploit_prob_annual: Annual probability of smart contract exploit
        exploit_loss: Fraction of position lost in exploit (0-1)
    
    Returns:
        Adjusted VaR incorporating DeFi risks
    """
    # 1. Liquidation cascade adjustment
    liquidation_var = normal_var * liquidation_multiplier
    
    # 2. Smart contract exploit risk (daily equivalent)
    exploit_prob_daily = exploit_prob_annual / 365
    exploit_var_daily = exploit_prob_daily * exploit_loss
    
    # Total adjusted VaR (additive for tail risks)
    total_var = liquidation_var + exploit_var_daily
    
    return {
        'Market VaR': normal_var,
        'Liquidation-Adjusted VaR': liquidation_var,
        'Exploit VaR (daily)': exploit_var_daily,
        'Total DeFi VaR': total_var,
    }


# Example: $1M DeFi portfolio
portfolio_value = 1_000_000
normal_daily_var = 0.025  # 2.5% daily VaR from market risk

defi_result = defi_var_adjustment(
    normal_var=normal_daily_var,
    liquidation_multiplier=1.8,  # High leverage portfolio
    exploit_prob_annual=0.005,   # 0.5% annual exploit risk
    exploit_loss=1.0             # Total loss if exploited
)

print("\n" + "="*60)
print("DeFi VaR ADJUSTMENT")
print("="*60)

print(f"\nPortfolio Value: ${portfolio_value:,.0f}")
print(f"\n{'Risk Component':<35} {'Daily VaR':<15} {'Dollar VaR':<15}")
print("-" * 65)
print(f"{'Market Risk (Normal VaR)':<35} {defi_result['Market VaR']:>8.2%}    ${defi_result['Market VaR'] * portfolio_value:>12,.0f}")
print(f"{'Liquidation Cascade Risk':<35} {defi_result['Liquidation-Adjusted VaR']:>8.2%}    ${defi_result['Liquidation-Adjusted VaR'] * portfolio_value:>12,.0f}")
print(f"{'Smart Contract Exploit (daily)':<35} {defi_result['Exploit VaR (daily)']:>8.4%}    ${defi_result['Exploit VaR (daily)'] * portfolio_value:>12,.0f}")
print("-" * 65)
print(f"{'Total DeFi-Adjusted VaR':<35} {defi_result['Total DeFi VaR']:>8.2%}    ${defi_result['Total DeFi VaR'] * portfolio_value:>12,.0f}")

print(f"\n>>> Market VaR alone: ${normal_daily_var * portfolio_value:,.0f}/day")
print(f">>> DeFi-adjusted:    ${defi_result['Total DeFi VaR'] * portfolio_value:,.0f}/day")
print(f">>> Underestimation:  {((defi_result['Total DeFi VaR'] - normal_daily_var) / normal_daily_var) * 100:.0f}%")

print("\n>>> CVaR for exploit risk: $1,000,000 (total loss)")
print(">>> Standard VaR models completely miss this tail!")

## 9. VaR Properties: VaR vs CVaR Comparison

In [ ]:
# Summarize the key differences between VaR and CVaR
print("\n" + "="*60)
print("VaR vs CVaR: Properties Comparison")
print("="*60)

properties = [
    ("Threshold metric", "Yes", "No"),
    ("Tail severity", "No", "Yes"),
    ("Coherent risk measure", "No", "Yes"),
    ("Subadditivity", "No (can fail)", "Yes (always)"),
    ("Monotonicity", "Yes", "Yes"),
    ("Positive homogeneity", "Yes", "Yes"),
    ("Translation invariance", "Yes", "Yes"),
]

print(f"\n{'Property':<30} {'VaR':<20} {'CVaR':<20}")
print("-" * 70)
for prop, var_val, cvar_val in properties:
    print(f"{prop:<30} {var_val:<20} {cvar_val:<20}")

print("\n" + "="*60)
print("What is Coherence?")
print("="*60)
print("""
A coherent risk measure satisfies four axioms:

1. Monotonicity: If X ≤ Y, then risk(X) ≥ risk(Y)
2. Subadditivity: risk(X + Y) ≤ risk(X) + risk(Y) [diversification]
3. Positive homogeneity: risk(λX) = λ·risk(X) for λ > 0
4. Translation invariance: risk(X + c) = risk(X) - c

CVaR satisfies ALL four. VaR FAILS subadditivity — the VaR of a
combined portfolio can exceed the sum of individual VaRs, violating
diversification logic.

This is why Basel III now requires BOTH VaR and Expected Shortfall.
""")

## Key Takeaways

1. **Three VaR methods:** Historical (non-parametric), Normal (fast but optimistic), Student-t (fat-tail aware, recommended).

2. **Always report CVaR alongside VaR.** VaR tells you the threshold; CVaR tells you the tail severity.

3. **√T scaling is approximate.** It fails during volatility clustering — use GARCH for multi-day VaR in stressed markets.

4. **Component VaR reveals concentration risk.** A portfolio can be diversified by weights but concentrated by risk contribution.

5. **Backtest your VaR model.** Use the Kupiec test to validate exception rates.

6. **DeFi requires VaR adjustments.** Liquidation cascade risk and smart contract exploit risk are not captured by standard models.

---

**References:**

- Jorion, P. (2007). *Value at Risk: The New Benchmark for Managing Financial Risk* (3rd ed.). McGraw-Hill.
- Artzner, P. et al. (1999). "Coherent Measures of Risk." *Mathematical Finance*, 9(3), 203–228.
- Kupiec, P.H. (1995). "Techniques for Verifying the Accuracy of Risk Measurement Models." *JOD*, 3(2), 73–84.

---
*Quantifaya — Quantitative Finance for Web2 & Web3. Not financial advice.*